In [2]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [3]:
#Cohort_Lab
Final_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Lab_Paired_AdditionalCols_Finalized.parquet")

In [3]:
Final_Cohort_Lab.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- refLowtype: string (nullable = true)
 |-- refLowRange: string (nullable = true)
 |-- refHightype: string (nullable = true)
 |-- refHighRange: string (nullable = true)



In [17]:
#Control_Lab
Final_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Lab_Paired_AdditionalCols_Finalized.parquet")

In [5]:
Final_Control_Lab.printSchema()

▸,:,


root
 |-- value: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- refLowtype: string (nullable = true)
 |-- refLowRange: string (nullable = true)
 |-- refHightype: string (nullable = true)
 |-- refHighRange: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [4]:
# case 1: There were cases where the interpretation is null/unknown/Not Applicable and reference range is also null/unknown/Not Applicable but there is a value. How do I interpret them.
# Cohort
from pyspark.sql.functions import col

# Calculate total number of records in Final_Cohort_Lab
total_records = Final_Cohort_Lab.count()
# Filter records based on specified conditions
filtered_Cohort_df = Final_Cohort_Lab.filter(
    (col("interpretation").isNull() |
     col("interpretation").isin("", "Unknown", "Not applicable")) &
    (col("refLowtype").isNull() |
     col("refLowtype").isin("", "Unknown", "Not applicable")) &
    (col("refLowRange").isNull() |
     col("refLowRange").isin("", "Unknown", "Not applicable")) &
    (col("refHightype").isNull() |
     col("refHightype").isin("", "Unknown", "Not applicable")) &
    (col("refHighRange").isNull() |
     col("refHighRange").isin("", "Unknown", "Not applicable")) &
#     (~(col("interpretation").isNull() | 
#       col("interpretation").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refLowtype").isNull() | 
#       col("refLowtype").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refLowRange").isNull() | 
#       col("refLowRange").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refHightype").isNull() | 
#       col("refHightype").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refHighRange").isNull() | 
#       col("refHighRange").isin("", "Unknown", "Not applicable"))) &
    ((col("value").isNull() |
      col("value").isin("", "Unknown", "Not applicable")) == False)
)

filtered_Cohort_df.show(20, truncate=False)
# Count the number of records
count_of_filtered_records  = filtered_Cohort_df.count()
# Calculate the percentage of records in filtered_Cohort_df
percentage_filtered = (count_of_filtered_records / total_records) * 100

print("Total number of records in Final_Cohort_Lab:", total_records)
print("Number of records satisfying the conditions in filtered_Cohort_df:", count_of_filtered_records)
print("Percentage of records satisfying the conditions:", percentage_filtered)

+------------------------------------+-------+----------+--------------+-------------------------+-----+--------+----------+-----------+-----------+------------+
|personid                            |labcode|loincclass|interpretation|servicedate              |value|modifier|refLowtype|refLowRange|refHightype|refHighRange|
+------------------------------------+-------+----------+--------------+-------------------------+-----+--------+----------+-----------+-----------+------------+
|877213a5-d7e9-4bf0-a9d0-c126158a15fa|736-9  |HEM/BC    |null          |2017-09-19T15:46:00+00:00|23.9 |null    |null      |null       |null       |null        |
|09cb771f-6857-4e12-aa8f-2b1523432bf2|713-8  |HEM/BC    |Not applicable|2021-04-21T16:55:00+00:00|0    |null    |null      |null       |null       |null        |
|9cf35508-4b4c-476d-8ce8-cfb089bf8663|706-2  |HEM/BC    |Not applicable|2020-01-24T04:15:00+00:00|0    |null    |null      |null       |null       |null        |
|2982ff62-4e22-463a-a036-925

In [5]:
# Subtract df2 from df1
result_Cohort_df = Final_Cohort_Lab.subtract(filtered_Cohort_df)

In [8]:
result_Cohort_df.show(20)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+---------+----------+-----------------+--------------------+-----+--------+----------+-----------+-----------+------------+
|            personid|  labcode|loincclass|   interpretation|         servicedate|value|modifier|refLowtype|refLowRange|refHightype|refHighRange|
+--------------------+---------+----------+-----------------+--------------------+-----+--------+----------+-----------+-----------+------------+
|0001beb3-781c-4cb...|   1963-8|      CHEM|           Normal|2019-12-08T08:36:...|   22|    null|   NUMERIC|         21|    NUMERIC|          31|
|0001beb3-781c-4cb...|    786-4|    HEM/BC|           Normal|2019-12-08T08:36:...| 33.9|    null|   NUMERIC|       33.0|    NUMERIC|        37.0|
|00266a24-ef77-44a...|   5769-5|        UA|           Normal|2019-08-18T23:23:...| null|    null|      TEXT|       null|       null|        null|
|00317af4-7a50-472...|  31418-7|     MICRO|           Normal|2014-03-05T14:47:...| null|    null|      TEXT|       null|    

<IPython.core.display.Javascript object>

In [18]:
# case 1: There were cases where the interpretation is null/unknown/Not Applicable and reference range is also null/unknown/Not Applicable but there is a value. How do I interpret them.
# Cohort
from pyspark.sql.functions import col

# Calculate total number of records in Final_Cohort_Lab
total_records = Final_Control_Lab.count()
# Filter records based on specified conditions
filtered_Control_df = Final_Control_Lab.filter(
    (col("interpretation").isNull() |
     col("interpretation").isin("", "Unknown", "Not applicable")) &
    (col("refLowtype").isNull() |
     col("refLowtype").isin("", "Unknown", "Not applicable")) &
    (col("refLowRange").isNull() |
     col("refLowRange").isin("", "Unknown", "Not applicable")) &
    (col("refHightype").isNull() |
     col("refHightype").isin("", "Unknown", "Not applicable")) &
    (col("refHighRange").isNull() |
     col("refHighRange").isin("", "Unknown", "Not applicable")) &
#     (~(col("interpretation").isNull() | 
#       col("interpretation").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refLowtype").isNull() | 
#       col("refLowtype").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refLowRange").isNull() | 
#       col("refLowRange").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refHightype").isNull() | 
#       col("refHightype").isin("", "Unknown", "Not applicable"))) &
#     (~(col("refHighRange").isNull() | 
#       col("refHighRange").isin("", "Unknown", "Not applicable"))) &
    ((col("value").isNull() |
      col("value").isin("", "Unknown", "Not applicable")) == False)
)

filtered_Control_df.show(20, truncate=False)
# Count the number of records
count_of_filtered_records  = filtered_Control_df.count()
# Calculate the percentage of records in filtered_Cohort_df
percentage_filtered = (count_of_filtered_records / total_records) * 100

print("Total number of records in Final_Cohort_Lab:", total_records)
print("Number of records satisfying the conditions in filtered_Cohort_df:", count_of_filtered_records)
print("Percentage of records satisfying the conditions:", percentage_filtered)

+------+--------+----------+-----------+-----------+------------+------------------------------------+---------+--------------+-------------------------+
|value |modifier|refLowtype|refLowRange|refHightype|refHighRange|personid                            |labcode  |interpretation|servicedate              |
+------+--------+----------+-----------+-----------+------------+------------------------------------+---------+--------------+-------------------------+
|-0.004|null    |null      |null       |null       |null        |969643bf-86ee-4493-b395-baa004c6e219|64084-7  |Not applicable|2020-10-21T14:15:00+00:00|
|-0.016|null    |null      |null       |null       |null        |23661a9c-5562-4176-b14c-7c55880c0d2e|88517-8  |Not applicable|2022-05-05T15:20:00+00:00|
|-0.1  |null    |null      |null       |null       |null        |02dba96d-635c-4557-b072-6f622015137d|6412-1   |null          |2021-09-07T23:25:00+00:00|
|-0.1  |null    |null      |null       |null       |null        |59bd5fab-44

In [19]:
# Subtract df2 from df1
result_Control_df = Final_Control_Lab.subtract(filtered_Control_df)

In [9]:
result_Cohort_df.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- refLowtype: string (nullable = true)
 |-- refLowRange: string (nullable = true)
 |-- refHightype: string (nullable = true)
 |-- refHighRange: string (nullable = true)



In [8]:
from pyspark.sql.functions import when

result_Cohort_df = result_Cohort_df.withColumn(
    "updated_interpretation",
    when(result_Cohort_df["value"] <= result_Cohort_df["refLowRange"], "low")
    .when(result_Cohort_df["value"] >= result_Cohort_df["refHighRange"], "high")
    .otherwise("Medium")
)

result_Cohort_df.show(20, truncate=False)

+------------------------------------+---------+----------+-----------------+-------------------------+-----+--------+----------+-----------+-----------+------------+----------------------+
|personid                            |labcode  |loincclass|interpretation   |servicedate              |value|modifier|refLowtype|refLowRange|refHightype|refHighRange|updated_interpretation|
+------------------------------------+---------+----------+-----------------+-------------------------+-----+--------+----------+-----------+-----------+------------+----------------------+
|0001beb3-781c-4cb3-9203-ebdcb389310e|1963-8   |CHEM      |Normal           |2019-12-08T08:36:00+00:00|22   |null    |NUMERIC   |21         |NUMERIC    |31          |Medium                |
|0001beb3-781c-4cb3-9203-ebdcb389310e|786-4    |HEM/BC    |Normal           |2019-12-08T08:36:00+00:00|33.9 |null    |NUMERIC   |33.0       |NUMERIC    |37.0        |Medium                |
|00266a24-ef77-44a1-9af9-7d412ac23f39|5769-5   |UA

In [20]:
from pyspark.sql.functions import when

result_Control_df = result_Control_df.withColumn(
    "updated_interpretation",
    when(result_Control_df["value"] <= result_Control_df["refLowRange"], "low")
    .when(result_Control_df["value"] >= result_Control_df["refHighRange"], "high")
    .otherwise("Medium")
)

result_Control_df.show(20, truncate=False)

+-----+--------+----------+-----------+-----------+------------+------------------------------------+---------+--------------+-------------------------+----------------------+
|value|modifier|refLowtype|refLowRange|refHightype|refHighRange|personid                            |labcode  |interpretation|servicedate              |updated_interpretation|
+-----+--------+----------+-----------+-----------+------------+------------------------------------+---------+--------------+-------------------------+----------------------+
|null |null    |null      |null       |null       |null        |000058c9-4684-4a8a-9913-6a6ba01f8208|5778-6   |null          |2022-04-05T16:45:00+00:00|Medium                |
|null |null    |null      |null       |null       |null        |0002d725-ab79-4901-b071-b0da416f06de|5811-5   |null          |2019-05-01T14:04:00+00:00|Medium                |
|null |null    |null      |null       |null       |null        |00055cf7-4e59-488e-ac39-b28b260d2dc3|11268-0  |null     

In [11]:
result_Cohort_df.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- refLowtype: string (nullable = true)
 |-- refLowRange: string (nullable = true)
 |-- refHightype: string (nullable = true)
 |-- refHighRange: string (nullable = true)
 |-- updated_interpretation: string (nullable = false)



In [17]:
# from pyspark.sql.functions import col

# # Counting the number of records meeting the specified conditions
# count_null_records = result_Cohort_df.filter(
#     col("value").isNull() & col("refLowRange").isNull() & col("refHighRange").isNull()
# ).count()

# print("Number of records with NULL value, refLowRange, and refHighRange:", count_null_records)
from pyspark.sql.functions import col

# Total number of records
total_records = result_Cohort_df.count()

# Counting the number of records meeting the specified conditions
count_null_records = result_Cohort_df.filter(
    col("value").isNull() & col("refLowRange").isNull() & col("refHighRange").isNull()
).count()

# Printing the total number of records
print("Total number of records in result_Cohort_df:", total_records)

# Printing the number of records with NULL value, refLowRange, and refHighRange
print("Number of records with NULL value, refLowRange, and refHighRange:", count_null_records)

# Computing and printing the percentage of count_null_records from the total count
percentage_null_records = (count_null_records / total_records) * 100
print("Percentage of count_null_records from the total count: {:.2f}%".format(percentage_null_records))

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of records in result_Cohort_df: 17081354
Number of records with NULL value, refLowRange, and refHighRange: 3029232
Percentage of count_null_records from the total count: 17.73%


<IPython.core.display.Javascript object>

In [16]:
from pyspark.sql.functions import col

# Counting the number of records meeting the specified conditions
count_null_records = result_Cohort_df.filter(
    col("value").isNull() & col("refLowRange").isNull() & col("refHighRange").isNull() & (col("updated_interpretation") != "Medium")
).count()

print("Number of records with NULL value, refLowRange, refHighRange, and updated_interpretation not equal to 'Medium':", count_null_records)


▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of records with NULL value, refLowRange, refHighRange, and updated_interpretation not equal to 'Medium': 0


<IPython.core.display.Javascript object>

In [13]:
from pyspark.sql.functions import col

# Counting the number of records meeting the specified conditions
count_null_records = result_Cohort_df.filter(
    col("value").isNull() & col("refLowRange").isNull() & col("refHighRange").isNull() & col("updated_interpretation") != "Medium"
).show(20, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+-----------+--------------+-------------------------+-----+--------+----------+-----------+-----------+------------+----------------------+
|personid                            |labcode|loincclass |interpretation|servicedate              |value|modifier|refLowtype|refLowRange|refHightype|refHighRange|updated_interpretation|
+------------------------------------+-------+-----------+--------------+-------------------------+-----+--------+----------+-----------+-----------+------------+----------------------+
|fc55db64-736a-4804-8b5a-623d9ae488e0|8172-9 |DRUG/TOX   |Abnormal      |2016-02-11T04:36:00+00:00|null |null    |TEXT      |null       |null       |null        |Medium                |
|00777776-bd91-4195-b0fe-8c8501ecf357|5778-6 |SPEC       |null          |2017-12-08T07:00:00+00:00|null |null    |null      |null       |null       |null        |Medium                |
|a6141e89-b107-4bbe-8ad0-c8f6a52ff337|8247-9 |UA         |Abnormal    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from pyspark.sql.functions import col

# # Filtering records with interpretation as Positive or Negative
# positive_negative_records = result_Cohort_df.filter(
#     (col("interpretation") == "Positive") | (col("interpretation") == "Negative") & (col("refLowtype") == "TEXT") | (col("refHightype") == "TEXT") & (col("updated_interpretation") != "Medium")
#     & (col("refLowRange") == "Positive") | (col("refLowRange") == "Negative") & (col("refHighRange") == "Positive") | (col("refHighRange") == "Negative")
# )
# Filtering records with interpretation as Positive or Negative
positive_negative_records = result_Cohort_df.filter(
   (col("interpretation") == "Positive") & (col("refHightype") == "TEXT") & (col("refHighRange") == "Positive") 
)
# Displaying the selected records
positive_negative_records.show(truncate=False)

+--------+-------+----------+--------------+-----------+-----+--------+----------+-----------+-----------+------------+----------------------+
|personid|labcode|loincclass|interpretation|servicedate|value|modifier|refLowtype|refLowRange|refHightype|refHighRange|updated_interpretation|
+--------+-------+----------+--------------+-----------+-----+--------+----------+-----------+-----------+------------+----------------------+
+--------+-------+----------+--------------+-----------+-----+--------+----------+-----------+-----------+------------+----------------------+



In [21]:
from pyspark.sql.functions import col
# Filtering records with interpretation as Positive or Negative
positive_negative_records = result_Control_df.filter(
   (col("interpretation") == "Positive") & (col("refHightype") == "TEXT") & (col("refHighRange") == "Positive") 
)
# Displaying the selected records
positive_negative_records.show(truncate=False)

+-----+--------+----------+-----------+-----------+------------+--------+-------+--------------+-----------+----------------------+
|value|modifier|refLowtype|refLowRange|refHightype|refHighRange|personid|labcode|interpretation|servicedate|updated_interpretation|
+-----+--------+----------+-----------+-----------+------------+--------+-------+--------------+-----------+----------------------+
+-----+--------+----------+-----------+-----------+------------+--------+-------+--------------+-----------+----------------------+

